# Radiology RVU Forecasting (3-Week Moving Average)

This Databricks notebook predicts hourly RVU using a 3-week moving average for each **(Modified_Clario_Site_ID, Priority)** combination.

**Source table**: `edw_dev.matrix_lateetud.exam_data_lateetud_deduped`


## 1) Runtime Parameters
Use widgets to set the prediction date and output CSV path.


In [ ]:
# Databricks widgets
try:
    dbutils.widgets.removeAll()
except Exception:
    pass

dbutils.widgets.text("prediction_date", "2026-03-22", "Prediction Date (YYYY-MM-DD)")
dbutils.widgets.text("output_path", "dbfs:/FileStore/rvu_predictions/rvu_forecast.csv", "Output CSV Path")

prediction_date = dbutils.widgets.get("prediction_date").strip()
output_path = dbutils.widgets.get("output_path").strip()

print(f"prediction_date = {prediction_date}")
print(f"output_path     = {output_path}")


## 2) Imports and Configuration


In [ ]:
import logging
import os
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Tuple

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("Radiology3WeekForecast")


## 3) Forecasting Class
Implements data loading, validation, 3-week lookup-based forecasting, metrics, and CSV export.


In [ ]:
@dataclass
class ForecastSummary:
    prediction_date: str
    training_start: str
    training_end: str
    test_start: str
    test_end: str
    training_rows: int
    test_rows: int
    output_rows: int
    coverage_any_history_pct: float
    coverage_full_history_pct: float
    mae: float
    rmse: float
    mape_pct: float


class Radiology3WeekForecast:
    """3-week moving-average forecaster for radiology RVU demand in Databricks."""

    SOURCE_TABLE = "edw_dev.matrix_lateetud.exam_data_lateetud_deduped"
    SITE_FILTER = (1297, 1298, 1299, 1300, 1301, 1302, 1303, 1304)
    REQUIRED_COLUMNS = ["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS", "RVU"]
    # Actual storage format of Modified_Unread_DTS in the source table.
    # TO_TIMESTAMP without an explicit pattern returns NULL for AM/PM strings.
    # Primary format (h handles both single- and double-digit 12-hour clock):
    TIMESTAMP_FORMAT = "yyyy-MM-dd h:mm a"
    # Fallback formats tried in order via COALESCE(TRY_TO_TIMESTAMP(...))
    TIMESTAMP_FORMATS = ("yyyy-MM-dd h:mm a", "yyyy-MM-dd hh:mm a")

    def __init__(self, prediction_date: str, output_path: str):
        # CORRECTED policy uses the modern java.time parser, which is strict but
        # correct.  Without this setting Databricks may raise
        # SparkUpgradeException: INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER
        try:
            spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
        except Exception:
            pass  # not running in Spark context (unit tests, etc.)
        self.prediction_date = prediction_date
        self.output_path = output_path
        self.prediction_start = self._parse_date(prediction_date)
        self.prediction_end = self.prediction_start + timedelta(days=1)
        self.training_start = self.prediction_start - timedelta(days=21)
        self.training_end = self.prediction_start
        self.train_pdf = None
        self.test_pdf = None
        self.results_pdf = None
        self.summary = None

    @staticmethod
    def _parse_date(date_text: str) -> datetime:
        try:
            return datetime.strptime(date_text, "%Y-%m-%d")
        except ValueError as e:
            raise ValueError("prediction_date must be in YYYY-MM-DD format") from e

    @staticmethod
    def _to_local_dbfs_path(path: str) -> str:
        if path.startswith("dbfs:/"):
            return "/dbfs/" + path[len("dbfs:/"):].lstrip("/")
        return path

    def _validate_table_schema(self):
        logger.info("Validating source table schema...")
        actual_cols = set(spark.table(self.SOURCE_TABLE).columns)
        missing = [c for c in self.REQUIRED_COLUMNS if c not in actual_cols]
        if missing:
            raise ValueError(f"Missing required columns in {self.SOURCE_TABLE}: {missing}")

    def _validate_column_types(self):
        """Log actual DataTypes of required columns to catch silent type mismatches early."""
        schema = {f.name: str(f.dataType) for f in spark.table(self.SOURCE_TABLE).schema.fields}
        for col in self.REQUIRED_COLUMNS:
            logger.info("Column %s -> %s", col, schema.get(col, "NOT FOUND"))

    @property
    def _parse_expr(self) -> str:
        """Return a SQL expression that safely parses Modified_Unread_DTS.

        Uses COALESCE(TRY_TO_TIMESTAMP(...), ...) across all TIMESTAMP_FORMATS so
        that single-digit hours (e.g. "4:30 PM") and double-digit hours (e.g.
        "04:30 PM") are both handled, and unparseable rows produce NULL instead of
        raising a hard job failure.
        """
        tries = ", ".join(
            f"TRY_TO_TIMESTAMP(Modified_Unread_DTS, '{fmt}')"
            for fmt in self.TIMESTAMP_FORMATS
        )
        return f"COALESCE({tries})"

    def _validate_timestamp_parsing(self):
        """Warn when a high fraction of rows cannot be parsed by any known format."""
        try:
            parse_expr = self._parse_expr
            row = spark.sql(f"""
                SELECT
                    COUNT(*) AS total,
                    SUM(CASE WHEN {parse_expr} IS NULL THEN 1 ELSE 0 END) AS null_count
                FROM {self.SOURCE_TABLE}
            """).collect()[0]
            total = row["total"] or 0
            null_count = row["null_count"] or 0
            if total > 0:
                null_pct = null_count / total * 100
                if null_pct > 10:
                    logger.warning(
                        "%.1f%% of rows (%d / %d) could not be parsed by formats %s. "
                        "Check TIMESTAMP_FORMATS for new data variants.",
                        null_pct, null_count, total, self.TIMESTAMP_FORMATS,
                    )
                else:
                    logger.info("Timestamp parse quality: %.1f%% NULL (out of %d rows)", null_pct, total)
        except Exception as exc:
            logger.warning("Could not validate timestamp parsing quality: %s", exc)

    def _validate_data_availability(self):
        """Query the actual MIN/MAX timestamps in the table and log the available range.

        This surfaces data-staleness problems early and makes the empty-training-data
        error actionable: users can see immediately which prediction_date values are valid.
        Returns (min_dt, max_dt) as Python datetime objects (or None if the table is empty).
        """
        logger.info("Checking available data range in source table...")
        try:
            parse_expr = self._parse_expr
            row = spark.sql(f"""
                SELECT
                    MIN({parse_expr}) AS min_ts,
                    MAX({parse_expr}) AS max_ts
                FROM {self.SOURCE_TABLE}
            """).collect()[0]
        except Exception as exc:
            logger.warning("Could not determine available data range: %s", exc)
            return None, None

        def _coerce(val):
            if val is None:
                return None
            if hasattr(val, "date"):
                return val
            for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"):
                try:
                    return datetime.strptime(str(val).split(".")[0], fmt)
                except ValueError:
                    continue
            return None

        min_dt = _coerce(row["min_ts"])
        max_dt = _coerce(row["max_ts"])

        if min_dt and max_dt:
            logger.info("Available data range: %s  →  %s", min_dt.date(), max_dt.date())
            logger.info(
                "Requested training window: %s  →  %s",
                self.training_start.date(), self.training_end.date(),
            )
        else:
            logger.warning("Source table appears to be empty or timestamps could not be parsed.")

        return min_dt, max_dt

    def load_data(self):
        logger.info("Loading training/test data from Spark SQL...")
        self._validate_table_schema()
        self._validate_column_types()
        self._validate_timestamp_parsing()
        self._available_min, self._available_max = self._validate_data_availability()

        # Use string literals so the IN filter is safe regardless of whether
        # Modified_Clario_Site_ID is stored as StringType, IntegerType, or DecimalType.
        site_csv_str = ",".join(f"'{s}'" for s in self.SITE_FILTER)

        train_start = f"{self.training_start:%Y-%m-%d %H:%M:%S}"
        train_end   = f"{self.training_end:%Y-%m-%d %H:%M:%S}"
        pred_start  = f"{self.prediction_start:%Y-%m-%d %H:%M:%S}"
        pred_end    = f"{self.prediction_end:%Y-%m-%d %H:%M:%S}"

        # TO_TIMESTAMP with an explicit format string avoids session-timezone and
        # dialect-parsing ambiguity across Databricks runtime versions.
        # CAST(RVU AS DOUBLE) prevents silent truncation from DecimalType arithmetic.
        parse_expr = self._parse_expr
        self.train_sql = f"""
        SELECT
            Modified_Clario_Site_ID,
            Priority,
            date_trunc('hour', {parse_expr})    AS Modified_Unread_DTS,
            SUM(COALESCE(CAST(RVU AS DOUBLE), 0.0))  AS RVU
        FROM {self.SOURCE_TABLE}
        WHERE CAST(Modified_Clario_Site_ID AS STRING) IN ({site_csv_str})
          AND {parse_expr} >= TO_TIMESTAMP('{train_start}', 'yyyy-MM-dd HH:mm:ss')
          AND {parse_expr} <  TO_TIMESTAMP('{train_end}',   'yyyy-MM-dd HH:mm:ss')
        GROUP BY 1, 2, 3
        """

        self.test_sql = f"""
        SELECT
            Modified_Clario_Site_ID,
            Priority,
            date_trunc('hour', {parse_expr})    AS Modified_Unread_DTS,
            SUM(COALESCE(CAST(RVU AS DOUBLE), 0.0))  AS RVU
        FROM {self.SOURCE_TABLE}
        WHERE CAST(Modified_Clario_Site_ID AS STRING) IN ({site_csv_str})
          AND {parse_expr} >= TO_TIMESTAMP('{pred_start}', 'yyyy-MM-dd HH:mm:ss')
          AND {parse_expr} <  TO_TIMESTAMP('{pred_end}',   'yyyy-MM-dd HH:mm:ss')
        GROUP BY 1, 2, 3
        """

        logger.debug("train_sql:\n%s", self.train_sql)
        logger.debug("test_sql:\n%s", self.test_sql)

        train_sdf = spark.sql(self.train_sql)
        test_sdf  = spark.sql(self.test_sql)
        logger.info("train rows (Spark count): %d", train_sdf.count())
        logger.info("test  rows (Spark count): %d", test_sdf.count())

        self.train_pdf = train_sdf.toPandas()
        self.test_pdf  = test_sdf.toPandas()

        for frame_name, pdf in [("train", self.train_pdf), ("test", self.test_pdf)]:
            if not pdf.empty:
                pdf["Modified_Unread_DTS"] = pd.to_datetime(pdf["Modified_Unread_DTS"])
                pdf["Priority"] = pdf["Priority"].astype(str)
            logger.info("%s rows: %s", frame_name, len(pdf))

        if self.train_pdf.empty:
            # Query the table for its latest timestamp so the user knows exactly
            # what prediction_date would be valid.
            # MAX() on a STRING column returns a plain str in Spark, so we must
            # parse it into a datetime before calling .date().
            try:
                latest_row = spark.sql(
                    f"SELECT MAX({self._parse_expr}) AS latest FROM {self.SOURCE_TABLE}"
                ).collect()
                latest_raw = latest_row[0]["latest"] if latest_row else None
                if latest_raw is None:
                    latest_dt = None
                elif hasattr(latest_raw, "date"):
                    # Spark returned a proper Timestamp / datetime object
                    latest_dt = latest_raw
                else:
                    # Spark returned a string — parse it defensively
                    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"):
                        try:
                            latest_dt = datetime.strptime(str(latest_raw).split(".")[0], fmt)
                            break
                        except ValueError:
                            continue
                    else:
                        latest_dt = None
            except Exception:
                latest_dt = None

            # Prefer the already-queried availability range; fall back to the MAX re-query.
            latest_dt = latest_dt or (self._available_max if hasattr(self, "_available_max") else None)
            earliest_dt = getattr(self, "_available_min", None)

            if latest_dt is not None:
                latest_date = latest_dt.date() if hasattr(latest_dt, "date") else latest_dt
                hint = (
                    f" Available data range: "
                    f"{earliest_dt.date() if earliest_dt and hasattr(earliest_dt, 'date') else 'unknown'}"
                    f" to {latest_date}."
                    f" Try a prediction_date between those dates."
                )
            else:
                hint = ""
            raise ValueError(
                "No training data found in the 21-day window. "
                f"Window: {self.training_start.date()} to {self.training_end.date()}.{hint}"
            )

        available_sites = sorted(self.train_pdf["Modified_Clario_Site_ID"].dropna().unique().tolist())
        missing_sites = sorted(set(self.SITE_FILTER) - set(available_sites))
        if missing_sites:
            logger.warning("Sites with no training data: %s", missing_sites)

        logger.info("Training window: %s -> %s", self.training_start, self.training_end)
        logger.info("Prediction day : %s", self.prediction_start.date())

    def compute_forecast(self):
        logger.info("Computing 3-week moving-average forecast...")

        train = self.train_pdf.copy()
        test = self.test_pdf.copy()

        combos = pd.concat([
            train[["Modified_Clario_Site_ID", "Priority"]],
            test[["Modified_Clario_Site_ID", "Priority"]],
        ], ignore_index=True).drop_duplicates().reset_index(drop=True)

        if combos.empty:
            raise ValueError("No (site, priority) combinations found for forecasting.")

        hours = pd.date_range(self.prediction_start, self.prediction_end - timedelta(hours=1), freq="h")
        base = combos.assign(_k=1).merge(pd.DataFrame({"Modified_Unread_DTS": hours, "_k": 1}), on="_k").drop(columns=["_k"])

        test_actual = (
            test.rename(columns={"RVU": "RVU_actual"})
            [["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS", "RVU_actual"]]
        )
        base = base.merge(test_actual, on=["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS"], how="left")

        lookup: Dict[Tuple[int, str, pd.Timestamp], float] = {
            (int(r.Modified_Clario_Site_ID), str(r.Priority), pd.Timestamp(r.Modified_Unread_DTS)): float(r.RVU)
            for r in train.itertuples(index=False)
        }

        lag_values = []
        for row in base.itertuples(index=False):
            site = int(row.Modified_Clario_Site_ID)
            priority = str(row.Priority)
            ts = pd.Timestamp(row.Modified_Unread_DTS)
            values = [
                lookup.get((site, priority, ts - pd.Timedelta(days=7)), np.nan),
                lookup.get((site, priority, ts - pd.Timedelta(days=14)), np.nan),
                lookup.get((site, priority, ts - pd.Timedelta(days=21)), np.nan),
            ]
            lag_values.append(values)

        lag_df = pd.DataFrame(lag_values, columns=["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"])
        out = pd.concat([base.reset_index(drop=True), lag_df], axis=1)
        out["history_points"] = out[["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"]].notna().sum(axis=1)
        out["RVU_predicted"] = out[["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"]].mean(axis=1)

        out = out.sort_values(["Modified_Unread_DTS", "Modified_Clario_Site_ID", "Priority"]).reset_index(drop=True)
        self.results_pdf = out

    def evaluate(self):
        logger.info("Calculating validation and summary metrics...")
        df = self.results_pdf.copy()

        coverage_any = float((df["history_points"] >= 1).mean() * 100.0) if len(df) else 0.0
        coverage_full = float((df["history_points"] == 3).mean() * 100.0) if len(df) else 0.0

        scored = df[df["RVU_actual"].notna() & df["RVU_predicted"].notna()].copy()
        if scored.empty:
            mae = rmse = mape = float("nan")
        else:
            err = scored["RVU_actual"] - scored["RVU_predicted"]
            mae = float(np.mean(np.abs(err)))
            rmse = float(np.sqrt(np.mean(np.square(err))))
            denom = scored["RVU_actual"].replace(0, np.nan)
            mape = float((np.abs(err / denom)).mean() * 100.0)

        self.summary = ForecastSummary(
            prediction_date=self.prediction_date,
            training_start=f"{self.training_start:%Y-%m-%d %H:%M:%S}",
            training_end=f"{self.training_end:%Y-%m-%d %H:%M:%S}",
            test_start=f"{self.prediction_start:%Y-%m-%d %H:%M:%S}",
            test_end=f"{self.prediction_end:%Y-%m-%d %H:%M:%S}",
            training_rows=int(len(self.train_pdf)),
            test_rows=int(len(self.test_pdf)),
            output_rows=int(len(df)),
            coverage_any_history_pct=coverage_any,
            coverage_full_history_pct=coverage_full,
            mae=mae,
            rmse=rmse,
            mape_pct=mape,
        )

        summary_df = pd.DataFrame([self.summary.__dict__])
        display(summary_df)

    def save_results(self):
        logger.info("Saving CSV output...")
        final_df = self.results_pdf[[
            "Modified_Clario_Site_ID",
            "Priority",
            "Modified_Unread_DTS",
            "RVU_actual",
            "RVU_predicted",
        ]].copy()

        local_path = self._to_local_dbfs_path(self.output_path)
        parent = os.path.dirname(local_path)

        if self.output_path.startswith("dbfs:/"):
            dbutils.fs.mkdirs(self.output_path.rsplit("/", 1)[0])
        if parent:
            os.makedirs(parent, exist_ok=True)

        final_df.to_csv(local_path, index=False)
        logger.info("Saved results to %s", self.output_path)
        display(final_df.head(20))

    def run(self):
        logger.info("Starting forecast pipeline...")
        self.load_data()
        self.compute_forecast()
        self.evaluate()
        self.save_results()
        logger.info("Forecast pipeline complete.")


## 4) Execute Forecast


In [ ]:
forecaster = Radiology3WeekForecast(prediction_date=prediction_date, output_path=output_path)
forecaster.run()


## 5) Optional Visualization
Actual vs predicted hourly RVU for combinations where actuals are available.


In [ ]:
import matplotlib.pyplot as plt

viz_df = forecaster.results_pdf.copy()
viz_df = viz_df[viz_df["RVU_actual"].notna()].copy()

if viz_df.empty:
    print("No actual values available on prediction day for charting.")
else:
    sample_combo = viz_df[["Modified_Clario_Site_ID", "Priority"]].drop_duplicates().iloc[0]
    site = int(sample_combo["Modified_Clario_Site_ID"])
    priority = str(sample_combo["Priority"])
    plot_df = viz_df[(viz_df["Modified_Clario_Site_ID"] == site) & (viz_df["Priority"] == priority)].copy()

    plt.figure(figsize=(12, 4))
    plt.plot(plot_df["Modified_Unread_DTS"], plot_df["RVU_actual"], marker="o", label="Actual")
    plt.plot(plot_df["Modified_Unread_DTS"], plot_df["RVU_predicted"], marker="o", label="Predicted")
    plt.title(f"Hourly RVU | Site {site} | Priority {priority}")
    plt.xlabel("Hour")
    plt.ylabel("RVU")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
